# MMU MiniMax H3 — 480p smoke test

同僚リポジトリのCUDA/Colab経路を参考に作成。**この環境で実生成未検証**。
初回は架空人物・正典素材を使わない「窓辺のコーヒーの湯気」1カット。
864×480 / 124 frames / 24fps / 約5.17秒 / seed 42 / base 20 steps。

1. AI Plusと同じアカウントで開き、残高・CU/時を記録。L4を選ぶ。
2. 最初のセルでGPUと配置地域を確認する。モデルはまだDLしない。
3. [現行ライセンス](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE)の対象地域と実行場所を照合。
   配置地域が取得できなければunknownとして記録し、動作確認を続ける。
4. 次のセルから順番に実行。地域情報は任意の記録項目。
5. 出力zipを回収してからランタイムを削除。失敗時も放置せずログを回収して削除。

このノートブックはDriveをマウントせず、全データは一時ランタイム内に置く。
画面を閉じる前に結果をDLすること。Google One増量・GCS・MiniMax APIキーは不要。


In [ ]:
#@title 1. GPU / region preflight (no model downloads)
import json, os, shutil, urllib.request
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Free GiB:', round(shutil.disk_usage('/content').free / 2**30, 1))
try:
    req = urllib.request.Request('http://metadata.google.internal/computeMetadata/v1/instance/zone',
                                 headers={'Metadata-Flavor': 'Google'})
    with urllib.request.urlopen(req, timeout=5) as r:
        DEPLOYMENT_ZONE = r.read().decode()
except Exception as e:
    DEPLOYMENT_ZONE = 'UNKNOWN: ' + type(e).__name__
print('Deployment zone:', DEPLOYMENT_ZONE)
print('Do not infer region from your account, browser locale or timezone.')


In [ ]:
#@title 2. Deployment evidence and work directory
TERRITORY_EVIDENCE = globals().get("DEPLOYMENT_ZONE", "unknown") # Informational; not a generation gate
from pathlib import Path
import subprocess, sys, time
ROOT = Path('/content/mmu-h3')
ROOT.mkdir(exist_ok=True)
STARTED = time.time()


In [ ]:
#@title 3. Write tools and test prompt
TOOLS = {'build_workflow.py': '#!/usr/bin/env python3\n"""Adapted for MMU (2026-10-26): 480p defaults, T2V and turbo support, validation.\nReference: sobaya-0141/Seedance_Madogiwa at 1d62fc1042a7a797e9dbc0fe0dc277bb4ec54cf5.\nBuild a MiniMax H3 API-format workflow JSON for one chapter.\n\nWiring is taken from ComfyUI\'s official local templates (video_minimax_h3_r2v.json /\nvideo_minimax_h3_i2v.json), converted to API format and then parameterized. Two\ncorrections vs. the shipped template, both required here:\n  * CLIPLoader uses a GPU-appropriate text encoder instead of the template\'s hardcoded\n    nvfp4_awq (which only runs natively on Blackwell). Override with --encoder.\n  * width/height/length are set literally instead of via ResolutionSelector +\n    ComfyMathExpression, so the frame count matches script.md exactly.\n\nusage:\n  build_h3_workflow.py --mode i2v --out ch1_workflow.json --prompt-file p.txt \\\n      --frames 294 --first ch1_start.png --last ch1_end.png [--width 1344 --height 768]\n  build_h3_workflow.py --mode r2v --out ch8_workflow.json --prompt-file p.txt \\\n      --frames 124 --image ch8_start.png --image ch8_end.png --image Fukuchan_sheet.png \\\n      --audio ch8_line1_fukuchan.wav\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\n\nENCODER = "qwen3vl_32b_minimax_h3_int8_convrot.safetensors"\nUNET_I2V = "minimax_h3_fl2va_pruned_int8_convrot.safetensors"\nUNET_R2V = "minimax_h3_ref2va_pruned_int8_convrot.safetensors"\nVIDEO_VAE = "minimax_h3_video_vae_fp16.safetensors"\nAUDIO_VAE = "minimax_h3_audio_vae_fp32.safetensors"\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--mode", choices=["t2v", "i2v", "r2v"], required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--prompt-file", required=True)\n    ap.add_argument("--frames", type=int, required=True)\n    ap.add_argument("--width", type=int, default=864)\n    ap.add_argument("--height", type=int, default=480)\n    ap.add_argument("--steps", type=int, default=20)\n    ap.add_argument("--seed", type=int, default=42)\n    ap.add_argument("--prefix", default="video/MiniMax_H3")\n    ap.add_argument("--first")\n    ap.add_argument("--last")\n    ap.add_argument("--image", action="append", default=[])\n    ap.add_argument("--audio", action="append", default=[])\n    ap.add_argument("--ref-image-size", default="match", choices=["match", "max"])\n    # Weights differ by GPU architecture — see RUNBOOK.md. Defaults are the INT8\n    # pair, which works on any CUDA card with INT8 tensor cores (Ampere and newer).\n    ap.add_argument("--encoder", default=ENCODER)\n    ap.add_argument("--unet-i2v", default=UNET_I2V)\n    ap.add_argument("--unet-r2v", default=UNET_R2V)\n    ap.add_argument("--turbo", action="store_true", help="I2V/T2V 8-step LoRA; R2V remains base 20 steps")\n    a = ap.parse_args()\n    if a.width < 32 or a.height < 32 or a.width % 32 or a.height % 32:\n        ap.error("width/height must be positive multiples of 32")\n    if not 5 <= a.frames <= 362 or a.steps < 1:\n        ap.error("frames must be 5..362 and steps positive")\n    if a.turbo and a.mode == "r2v":\n        ap.error("R2V turbo is not validated in this workflow")\n\n    if a.frames % 17 != 5:\n        raise SystemExit(f"--frames {a.frames} is not on H3\'s 17k+5 grid (e.g. 90, 124, 294)")\n    if a.mode == "r2v":\n        if not a.image:\n            raise SystemExit("r2v needs at least one --image")\n        if len(a.image) > 9:\n            raise SystemExit(f"{len(a.image)} images exceeds H3\'s limit of 9")\n        if len(a.audio) > 3:\n            raise SystemExit(f"{len(a.audio)} audio files exceeds H3\'s limit of 3")\n        if len(a.image) + len(a.audio) > 12:\n            raise SystemExit("more than 12 total input files")\n    elif a.mode == "i2v":\n        if not (a.first and a.last):\n            raise SystemExit("i2v needs --first and --last")\n\n    prompt = open(a.prompt_file, encoding="utf-8").read().strip()\n\n    g: dict[str, dict] = {\n        "119": {"class_type": "VAELoader", "inputs": {"vae_name": VIDEO_VAE}},\n        "120": {"class_type": "VAELoader", "inputs": {"vae_name": AUDIO_VAE}},\n        "127": {"class_type": "UNETLoader", "inputs": {\n            "unet_name": a.unet_r2v if a.mode == "r2v" else a.unet_i2v,\n            "weight_dtype": "default"}},\n        "128": {"class_type": "CLIPLoader", "inputs": {\n            "clip_name": a.encoder, "type": "minimax", "device": "default"}},\n        "129": {"class_type": "RandomNoise", "inputs": {"noise_seed": a.seed}},\n        "123": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "res_multistep"}},\n        "124": {"class_type": "BasicScheduler", "inputs": {\n            "model": ["127", 0], "scheduler": "simple", "steps": a.steps, "denoise": 1}},\n        "126": {"class_type": "BasicGuider", "inputs": {\n            "model": ["127", 0], "conditioning": ["136", 0]}},\n        "125": {"class_type": "SamplerCustomAdvanced", "inputs": {\n            "noise": ["129", 0], "guider": ["126", 0], "sampler": ["123", 0],\n            "sigmas": ["124", 0], "latent_image": ["136", 1]}},\n        "122": {"class_type": "VAEDecode", "inputs": {"samples": ["125", 0], "vae": ["119", 0]}},\n        "121": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["125", 0], "vae": ["120", 0]}},\n        "130": {"class_type": "CreateVideo", "inputs": {\n            "images": ["122", 0], "audio": ["121", 0], "fps": 24, "bit_depth": 8}},\n        "92": {"class_type": "SaveVideo", "inputs": {\n            # codecはComfyUI新版(2026-08頃〜)で必須。旧版(v0.30)は未知の入力を無視するので常に付けてよい\n            "video": ["130", 0], "filename_prefix": a.prefix, "format": "auto", "codec": "auto"}},\n    }\n\n    if a.turbo:\n        g["150"] = {"class_type": "LoraLoaderModelOnly", "inputs": {\n            "model": ["127", 0], "lora_name": "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", "strength_model": 1.0}}\n        g["124"]["inputs"].update(model=["150", 0], steps=8)\n        g["126"]["inputs"]["model"] = ["150", 0]\n\n    cond: dict = {\n        "clip": ["128", 0],\n        "vae": ["119", 0],\n        "prompt": prompt,\n        "width": a.width,\n        "height": a.height,\n        "length": a.frames,\n    }\n\n    nid = 200\n    if a.mode == "i2v":\n        g[str(nid)] = {"class_type": "LoadImage", "inputs": {"image": a.first}}\n        cond["first_frame"] = [str(nid), 0]\n        nid += 1\n        g[str(nid)] = {"class_type": "LoadImage", "inputs": {"image": a.last}}\n        cond["last_frame"] = [str(nid), 0]\n        nid += 1\n        g["136"] = {"class_type": "MiniMaxH3ImageToVideo", "inputs": cond}\n    elif a.mode == "t2v":\n        g["136"] = {"class_type": "MiniMaxH3ImageToVideo", "inputs": cond}\n    else:\n        cond["audio_vae"] = ["120", 0]\n        cond["ref_image_size"] = a.ref_image_size\n        # Autogrow inputs are addressed by their dotted slot names, 0-indexed.\n        # Connection order defines the <Picture i> / <Audio j> tags in the prompt.\n        for i, img in enumerate(a.image):\n            g[str(nid)] = {"class_type": "LoadImage", "inputs": {"image": img}}\n            cond[f"ref_images.ref_image_{i}"] = [str(nid), 0]\n            nid += 1\n        for j, aud in enumerate(a.audio):\n            g[str(nid)] = {"class_type": "LoadAudio", "inputs": {"audio": aud}}\n            cond[f"ref_audios.ref_audio_{j}"] = [str(nid), 0]\n            nid += 1\n        g["136"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": cond}\n\n    with open(a.out, "w", encoding="utf-8") as fh:\n        json.dump(g, fh, ensure_ascii=False, indent=2)\n\n    secs = a.frames / 24\n    print(f"wrote {a.out}: mode={a.mode} {a.width}x{a.height} "\n          f"{a.frames}f ({secs:.3f}s @24fps) steps={8 if a.turbo else a.steps} seed={a.seed}")\n    if a.mode == "r2v":\n        for i, img in enumerate(a.image):\n            print(f"  <Picture {i+1}> = {img}")\n        for j, aud in enumerate(a.audio):\n            print(f"  <Audio {j+1}> = {aud}")\n    else:\n        print(f"  first_frame = {a.first}\\n  last_frame  = {a.last}")\n\n\nif __name__ == "__main__":\n    main()\n', 'setup_cuda.py': '#!/usr/bin/env python3\n"""Install pinned ComfyUI and only the weights needed by an H3 API workflow.\nRun on a dedicated Linux CUDA runtime, not on the author\'s Mac.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport urllib.request\n\nCOMFY_REV = \'fe298e4e20f1ff26d30e8639267bbd0b57811390\'\nHF_REV = \'e5eb578a89295337b8ff433a035929ce0279e0b6\'\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'workflow\', type=Path)\n    p.add_argument(\'--root\', type=Path, default=Path(\'/content/mmu-h3\'))\n    p.add_argument(\'--territory-evidence\', default=\'unknown\',\n                   help=\'Optional deployment note recorded in setup.json; unknown does not block execution\')\n    a = p.parse_args()\n    import torch\n    if sys.platform != \'linux\' or not torch.cuda.is_available():\n        p.error(\'A Linux NVIDIA CUDA runtime is required\')\n    cap = torch.cuda.get_device_capability(0)\n    if cap < (8, 0):\n        p.error(\'This setup targets Ampere or newer (L4/A100); T4 is not supported\')\n    a.root.mkdir(parents=True, exist_ok=True)\n    graph = json.loads(a.workflow.read_text())\n    for n in graph.values():\n        if n[\'class_type\'] == \'UNETLoader\':\n            name = n[\'inputs\'][\'unet_name\']\n            if cap >= (8, 9):\n                n[\'inputs\'][\'unet_name\'] = name.replace(\'int8_convrot\', \'fp8_scaled\')\n    runtime_wf = a.workflow.with_name(a.workflow.stem + \'.runtime.json\')\n    runtime_wf.write_text(json.dumps(graph, indent=2))\n    mapping = {\'UNETLoader\': (\'diffusion_models\', \'unet_name\'),\n               \'CLIPLoader\': (\'text_encoders\', \'clip_name\'),\n               \'VAELoader\': (\'vae\', \'vae_name\'),\n               \'LoraLoaderModelOnly\': (\'loras\', \'lora_name\')}\n    files = [f\'{mapping[n["class_type"]][0]}/{n["inputs"][mapping[n["class_type"]][1]]}\'\n             for n in graph.values() if n[\'class_type\'] in mapping]\n    if shutil.disk_usage(a.root).free < 60 * 2**30:\n        p.error(\'Provide at least 60 GiB free disk for the first install (~54 GB weights plus dependencies)\')\n    started = time.time()\n    comfy = a.root / \'ComfyUI\'\n    if not (comfy / \'.git\').exists():\n        subprocess.run([\'git\', \'clone\', \'--no-checkout\', \'--filter=blob:none\',\n                        \'https://github.com/Comfy-Org/ComfyUI.git\', str(comfy)], check=True)\n        subprocess.run([\'git\', \'checkout\', COMFY_REV], cwd=comfy, check=True)\n    actual = subprocess.check_output([\'git\', \'rev-parse\', \'HEAD\'], cwd=comfy, text=True).strip()\n    if actual != COMFY_REV:\n        p.error(\'Existing ComfyUI revision differs; use a new --root rather than overwrite it\')\n    subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-r\', str(comfy / \'requirements.txt\'),\n                    \'huggingface_hub\'], check=True)\n    from huggingface_hub import hf_hub_download\n    for name in files:\n        print(\'Downloading\', name, flush=True)\n        hf_hub_download(\'Comfy-Org/MiniMax-H3\', name, revision=HF_REV,\n                       local_dir=comfy / \'models\')\n    for n in graph.values():\n        key = {\'LoadImage\': \'image\', \'LoadAudio\': \'audio\'}.get(n[\'class_type\'])\n        if key:\n            name = n[\'inputs\'][key]\n            if Path(name).name != name:\n                p.error(\'Input references must use flat filenames\')\n            shutil.copy2(a.workflow.parent / name, comfy / \'input\' / name)\n    report = dict(gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda, torch=torch.__version__,\n                  comfy_revision=actual, weights_revision=HF_REV, model_files=files,\n                  territory_evidence=a.territory_evidence, setup_seconds=time.time() - started)\n    (a.root / \'setup.json\').write_text(json.dumps(report, indent=2))\n    log = a.root / \'comfy.log\'\n    with log.open(\'a\') as f:\n        proc = subprocess.Popen([sys.executable, \'main.py\', \'--listen\', \'127.0.0.1\', \'--port\', \'8188\'],\n                                cwd=comfy, stdout=f, stderr=subprocess.STDOUT, start_new_session=True)\n    (a.root / \'comfy.pid\').write_text(str(proc.pid))\n    for _ in range(120):\n        if proc.poll() is not None:\n            raise RuntimeError(f\'ComfyUI exited; inspect {log}\')\n        try:\n            with urllib.request.urlopen(\'http://127.0.0.1:8188/object_info\', timeout=5) as r:\n                info = json.load(r)\n            missing = {n[\'class_type\'] for n in graph.values()} - set(info)\n            if missing:\n                raise RuntimeError(f\'Missing nodes: {missing}\')\n            print(\'Ready:\', runtime_wf)\n            return\n        except (OSError, TimeoutError):\n            time.sleep(2)\n    raise TimeoutError(f\'ComfyUI startup timed out; inspect {log}\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'run_workflow.py': '#!/usr/bin/env python3\n"""Submit once, persist prompt_id, resume polling, and validate the downloaded MP4."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport time\nimport urllib.request\nimport urllib.parse\nimport uuid\n\n\ndef api(server, path, payload=None):\n    req = urllib.request.Request(server.rstrip(\'/\') + path,\n        data=None if payload is None else json.dumps(payload).encode(),\n        headers={\'Content-Type\': \'application/json\'})\n    with urllib.request.urlopen(req, timeout=60) as r:\n        return json.load(r)\n\n\ndef save(path, data):\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False) + \'\\n\')\n    tmp.replace(path)\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'workflow\', type=Path)\n    p.add_argument(\'--out\', required=True, type=Path)\n    p.add_argument(\'--server\', default=\'http://127.0.0.1:8188\')\n    p.add_argument(\'--timeout\', type=int, default=7200)\n    p.add_argument(\'--poll\', type=float, default=15)\n    a = p.parse_args()\n    if a.timeout <= 0 or a.poll <= 0:\n        p.error(\'timeout/poll must be positive\')\n    if not shutil.which(\'ffprobe\'):\n        p.error(\'ffprobe is required before submission\')\n    graph = json.loads(a.workflow.read_text())\n    digest = hashlib.sha256(a.workflow.read_bytes()).hexdigest()\n    a.out.parent.mkdir(parents=True, exist_ok=True)\n    state_path = a.out.with_suffix(\'.job.json\')\n    if state_path.exists():\n        state = json.loads(state_path.read_text())\n        if state[\'workflow_sha256\'] != digest or state[\'server\'] != a.server:\n            p.error(\'Existing job belongs to different inputs/server; choose a new output path\')\n        if state.get(\'status\') == \'verified\' and a.out.exists():\n            print(\'Already verified:\', a.out)\n            return\n        if state.get(\'status\') == \'submission_unknown\':\n            p.error(\'Submission response was lost. Inspect server queue/history; do not blindly resubmit.\')\n    else:\n        if a.out.exists():\n            p.error(\'Output already exists without job state; choose a new output path\')\n        api(a.server, \'/system_stats\')\n        state = dict(server=a.server, workflow_sha256=digest, client_id=uuid.uuid4().hex,\n                     started_unix=time.time(), status=\'submission_unknown\')\n        save(state_path, state)  # Do not retry a POST after a lost response.\n        result = api(a.server, \'/prompt\', {\'prompt\': graph, \'client_id\': state[\'client_id\']})\n        if result.get(\'error\') or not result.get(\'prompt_id\'):\n            state.update(status=\'rejected\', response=result)\n            save(state_path, state)\n            raise RuntimeError(result)\n        state.update(prompt_id=result[\'prompt_id\'], status=\'queued\')\n        save(state_path, state)\n    if not state.get(\'prompt_id\'):\n        p.error(\'No submitted prompt_id. Correct inputs and use a new output path.\')\n    print(\'prompt_id:\', state[\'prompt_id\'], flush=True)\n    deadline = time.monotonic() + a.timeout\n    while time.monotonic() < deadline:\n        entry = api(a.server, \'/history/\' + state[\'prompt_id\']).get(state[\'prompt_id\'])\n        if entry:\n            status = entry.get(\'status\', {})\n            if status.get(\'status_str\') == \'error\':\n                state.update(status=\'failed\', messages=status.get(\'messages\'))\n                save(state_path, state)\n                raise RuntimeError(state[\'messages\'])\n            if status.get(\'completed\'):\n                break\n        print(\'Waiting; elapsed seconds:\', round(time.time() - state[\'started_unix\']), flush=True)\n        time.sleep(a.poll)\n    else:\n        raise TimeoutError(\'Polling stopped; server job may still run. Resume this same command; do not resubmit.\')\n    videos = [f for output in entry.get(\'outputs\', {}).values()\n              for key in (\'videos\', \'gifs\', \'images\') for f in output.get(key, [])\n              if f.get(\'filename\', \'\').lower().endswith(\'.mp4\')]\n    if len(videos) != 1:\n        raise RuntimeError(f\'Expected one MP4, received {len(videos)}\')\n    item = videos[0]\n    query = urllib.parse.urlencode({k: item.get(k, \'output\' if k == \'type\' else \'\')\n                                   for k in (\'filename\', \'subfolder\', \'type\')})\n    part = a.out.with_suffix(\'.part.mp4\')\n    with urllib.request.urlopen(a.server.rstrip(\'/\') + \'/view?\' + query, timeout=600) as r, part.open(\'wb\') as f:\n        shutil.copyfileobj(r, f)\n    probe = json.loads(subprocess.check_output([\'ffprobe\', \'-v\', \'error\', \'-count_frames\',\n        \'-show_streams\', \'-show_format\', \'-of\', \'json\', str(part)]))\n    v = next(s for s in probe[\'streams\'] if s[\'codec_type\'] == \'video\')\n    cond = next(n[\'inputs\'] for n in graph.values() if n[\'class_type\'] in\n                (\'MiniMaxH3ImageToVideo\', \'MiniMaxH3ReferenceToVideo\'))\n    if (v[\'width\'], v[\'height\']) != (cond[\'width\'], cond[\'height\']):\n        raise RuntimeError(\'Output resolution mismatch\')\n    if int(v[\'nb_read_frames\']) != cond[\'length\'] or v[\'avg_frame_rate\'] != \'24/1\':\n        raise RuntimeError(\'Output frame count/fps mismatch\')\n    if not any(s[\'codec_type\'] == \'audio\' for s in probe[\'streams\']):\n        raise RuntimeError(\'Native audio stream missing\')\n    part.replace(a.out)\n    state.update(status=\'verified\', elapsed_seconds=round(time.time() - state[\'started_unix\'], 2),\n                 output_sha256=hashlib.sha256(a.out.read_bytes()).hexdigest(), probe=probe)\n    save(state_path, state)\n    print(\'MP4 structure verified (visual/audio review still required):\', a.out)\n\n\nif __name__ == \'__main__\':\n    main()\n'}
for name, source in TOOLS.items():
    (ROOT / name).write_text(source)
prompt = "A locked-off cinematic close-up of one white ceramic coffee cup on a wooden desk beside a sunlit office window. A thin wisp of steam rises and curls gently throughout the shot. The cup and camera remain still. Warm natural daylight, realistic materials. No people, no writing, no logos. Soundscape: quiet room tone. Music: none. Speech: none."
(ROOT / 'prompt.txt').write_text(prompt)
subprocess.run([sys.executable, str(ROOT/'build_workflow.py'), '--mode', 't2v',
    '--prompt-file', str(ROOT/'prompt.txt'), '--frames', '124', '--width', '864', '--height', '480',
    '--out', str(ROOT/'smoke.json')], check=True)


In [ ]:
#@title 4. Install and start ComfyUI (~54 GB download; consumes GPU runtime)
subprocess.run([sys.executable, str(ROOT/'setup_cuda.py'), str(ROOT/'smoke.json'),
    '--root', str(ROOT), '--territory-evidence', TERRITORY_EVIDENCE], check=True)


In [ ]:
#@title 5. Generate exactly one clip (resume with same job state; never blind retry)
subprocess.run([sys.executable, str(ROOT/'run_workflow.py'), str(ROOT/'smoke.runtime.json'),
    '--out', str(ROOT/'smoke_480p.mp4'), '--timeout', '7200'], check=True)


In [ ]:
#@title 6. Export 720p, preview and download evidence
# 864x480 is the model's 32-pixel grid. Preserve aspect ratio, pad to 1280x720.
subprocess.run(['ffmpeg', '-n', '-i', str(ROOT/'smoke_480p.mp4'), '-vf',
    'scale=1280:720:force_original_aspect_ratio=decrease:force_divisible_by=2:flags=lanczos,pad=1280:720:(ow-iw)/2:(oh-ih)/2,setsar=1',
    '-c:v', 'libx264', '-crf', '18', '-pix_fmt', 'yuv420p', '-c:a', 'copy', str(ROOT/'smoke_720p.mp4')], check=True)
(ROOT/'elapsed.json').write_text(json.dumps({'wall_seconds': time.time()-STARTED}))
from IPython.display import Video, display
display(Video(str(ROOT/'smoke_720p.mp4'), embed=True))
import zipfile
with zipfile.ZipFile('/content/mmu_h3_smoke_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ['smoke_480p.mp4','smoke_720p.mp4','smoke_480p.job.json','setup.json',
                 'elapsed.json','smoke.runtime.json','prompt.txt','comfy.log']:
        z.write(ROOT/name, name)
from google.colab import files
files.download('/content/mmu_h3_smoke_results.zip')
print('After verifying download: Runtime > Disconnect and delete runtime. Record ending CU balance.')
